In [1]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
import os
import re
from qdrant_client import QdrantClient, models
from qdrant_client.models import (
    VectorParams, Distance, PointStruct,
    Filter, FieldCondition, MatchValue, Range
)
from sentence_transformers import SentenceTransformer


client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))

# For Colab:
# from google.colab import userdata
# client = QdrantClient(url=userdata.get("QDRANT_URL"), api_key=userdata.get("QDRANT_API_KEY"))

encoder = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
my_dataset = [
    {
        "title": "Classic Beef Bourguignon",
        "description": """A rich, wine-braised beef stew from Burgundy, France. 
        Tender chunks of beef are slowly simmered with pearl onions, mushrooms, 
        and bacon in a deep red wine sauce. The long, slow cooking process 
        develops complex flavors and creates a luxurious, velvety texture. 
        Perfect for cold winter evenings when you want something hearty and 
        comforting. Traditionally served with crusty bread or creamy mashed 
        potatoes to soak up the incredible sauce.""",
        "cuisine": "French",
        "difficulty": "Intermediate",
        "time": "3 hours"
    },
    {
        "title": "Spaghetti Carbonara",
        "description": """A classic Roman pasta dish made with silky eggs, 
        sharp Pecorino Romano cheese, crispy guanciale, and freshly cracked 
        black pepper. The magic of carbonara comes from creating a creamy sauce 
        without using cream, relying instead on the heat of the pasta to gently 
        emulsify the eggs and cheese. It is simple, elegant, and deeply satisfying, 
        with a rich savory flavor balanced by the bite of pepper. Best served 
        immediately while the sauce is glossy and the pasta is perfectly al dente.""",
        "cuisine": "Italian",
        "difficulty": "Intermediate",
        "time": "30 minutes"
    },
    {
        "title": "Chicken Tikka Masala",
        "description": """A flavorful dish featuring tender pieces of marinated 
        chicken cooked in a creamy tomato-based curry sauce. The chicken is first 
        seasoned with yogurt, garlic, ginger, and warm spices, then grilled or 
        pan-seared until slightly charred. The sauce is rich, aromatic, and mildly 
        spicy, with layers of garam masala, cumin, coriander, and paprika. Served 
        with steamed basmati rice or warm naan, this dish is comforting, colorful, 
        and perfect for a cozy dinner.""",
        "cuisine": "Indian",
        "difficulty": "Intermediate",
        "time": "1 hour"
    },
    {
        "title": "Sushi Rolls",
        "description": """Fresh and delicate Japanese sushi rolls made with 
        seasoned rice, nori seaweed, crisp vegetables, and fillings such as 
        salmon, tuna, avocado, or cucumber. Each roll requires careful preparation, 
        from cooking the rice to the right sticky texture to slicing ingredients 
        into even strips. The final result is clean, balanced, and visually 
        beautiful, with a contrast of soft rice, fresh seafood, and crunchy 
        vegetables. Served with soy sauce, wasabi, and pickled ginger for a 
        refreshing meal or appetizer.""",
        "cuisine": "Japanese",
        "difficulty": "Advanced",
        "time": "1 hour 30 minutes"
    },
    {
        "title": "Vegetable Pad Thai",
        "description": """A popular Thai noodle dish filled with chewy rice 
        noodles, crisp vegetables, tofu, eggs, roasted peanuts, and a tangy-sweet 
        tamarind sauce. The dish is quickly stir-fried over high heat, creating 
        a lively combination of textures and flavors. Tamarind brings brightness, 
        palm sugar adds sweetness, and lime juice gives a fresh finish. It is 
        colorful, satisfying, and easy to customize with different vegetables or 
        proteins. A generous sprinkle of crushed peanuts makes every bite more 
        fragrant and crunchy.""",
        "cuisine": "Thai",
        "difficulty": "Intermediate",
        "time": "40 minutes"
    },
    {
        "title": "Shakshuka",
        "description": """A vibrant North African and Middle Eastern dish made 
        with eggs gently poached in a spicy tomato and pepper sauce. The sauce is 
        flavored with garlic, onions, cumin, paprika, and chili, creating a warm 
        and aromatic base. As the eggs cook, their yolks remain soft and creamy, 
        mixing beautifully with the rich tomato sauce. Shakshuka is often served 
        straight from the pan with crusty bread for dipping. It works wonderfully 
        as breakfast, brunch, or a simple weeknight dinner.""",
        "cuisine": "Middle Eastern",
        "difficulty": "Easy",
        "time": "35 minutes"
    },
    {
        "title": "Mexican Chicken Enchiladas",
        "description": """Soft corn tortillas filled with shredded chicken, 
        cheese, and spices, then rolled and baked under a rich chili sauce. The 
        filling is savory and comforting, while the sauce adds depth, warmth, 
        and a gentle smoky flavor. As the enchiladas bake, the cheese melts into 
        the filling and the edges of the tortillas become slightly crisp. Topped 
        with sour cream, cilantro, avocado, or fresh onions, this dish is hearty, 
        colorful, and ideal for sharing with friends or family.""",
        "cuisine": "Mexican",
        "difficulty": "Intermediate",
        "time": "1 hour"
    },
    {
        "title": "Greek Moussaka",
        "description": """A layered Greek casserole made with roasted eggplant, 
        seasoned minced meat, tomato sauce, and a creamy béchamel topping. The 
        eggplant becomes soft and slightly smoky after roasting, while the meat 
        sauce is flavored with cinnamon, garlic, oregano, and red wine. The final 
        layer of béchamel bakes into a golden, velvety crust. Moussaka is rich, 
        warming, and deeply flavorful, making it a perfect centerpiece for a 
        weekend meal or special family dinner.""",
        "cuisine": "Greek",
        "difficulty": "Advanced",
        "time": "2 hours"
    },
    {
        "title": "Korean Bibimbap",
        "description": """A colorful Korean rice bowl topped with seasoned 
        vegetables, marinated beef, a fried egg, and spicy gochujang sauce. Each 
        ingredient is prepared separately to preserve its own flavor and texture, 
        from crisp bean sprouts to earthy spinach and sweet carrots. Before eating, 
        everything is mixed together so the rice absorbs the sauce, egg yolk, and 
        savory juices. Bibimbap is balanced, nourishing, and visually striking, 
        offering a delicious combination of freshness, heat, and umami.""",
        "cuisine": "Korean",
        "difficulty": "Intermediate",
        "time": "1 hour"
    },
    {
        "title": "Moroccan Lamb Tagine",
        "description": """A slow-cooked Moroccan stew made with tender lamb, 
        dried fruits, warm spices, and aromatic vegetables. The lamb is gently 
        simmered with cinnamon, cumin, coriander, ginger, and saffron until it 
        becomes soft and deeply flavorful. Apricots or prunes add a natural 
        sweetness that balances the savory richness of the meat. Traditionally 
        cooked in a clay tagine, this dish develops a fragrant sauce that pairs 
        beautifully with couscous or flatbread. It is festive, comforting, and 
        full of layered spice.""",
        "cuisine": "Moroccan",
        "difficulty": "Intermediate",
        "time": "2 hours 30 minutes"
    },
    {
        "title": "American Apple Pie",
        "description": """A comforting dessert made with tender spiced apples 
        baked inside a flaky, buttery pie crust. The apple filling is usually 
        seasoned with cinnamon, nutmeg, sugar, and a touch of lemon juice to 
        create a warm balance of sweetness and acidity. As the pie bakes, the 
        crust turns golden and crisp while the apples become soft and syrupy. 
        Served warm with vanilla ice cream or whipped cream, apple pie is a 
        nostalgic dessert that feels especially perfect in autumn and winter.""",
        "cuisine": "American",
        "difficulty": "Intermediate",
        "time": "1 hour 30 minutes"
    }
]

In [3]:
def fixed_size_chunks(text, chunk_size=100, overlap=20):
    """Split text into fixed-size chunks with overlap"""
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk_words = words[i:i + chunk_size]
        if chunk_words:  # Only add non-empty chunks
            chunks.append(' '.join(chunk_words))
    
    return chunks

def sentence_chunks(text, max_sentences=3):
    """Group sentences into chunks"""
    import re
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk_sentences = sentences[i:i + max_sentences]
        if chunk_sentences:
            chunks.append('. '.join(chunk_sentences) + '.')
    
    return chunks

def paragraph_chunks(text):
    """Split by paragraphs or double line breaks"""
    chunks = [chunk.strip() for chunk in text.split('\n\n') if chunk.strip()]
    return chunks if chunks else [text]  # Fallback to full text

In [4]:
for i in range(len(my_dataset)):
    sample = my_dataset[i]["description"]
    print(f"\n Sample: {my_dataset[i]['title']}")
    print(f"   Fixed chunks:    {len(fixed_size_chunks(sample))} chunks")
    print(f"   Sentence chunks: {len(sentence_chunks(sample))} chunks")
    print(f"   Paragraph chunks:{len(paragraph_chunks(sample))} chunks")



 Sample: Classic Beef Bourguignon
   Fixed chunks:    1 chunks
   Sentence chunks: 2 chunks
   Paragraph chunks:1 chunks

 Sample: Spaghetti Carbonara
   Fixed chunks:    2 chunks
   Sentence chunks: 2 chunks
   Paragraph chunks:1 chunks

 Sample: Chicken Tikka Masala
   Fixed chunks:    1 chunks
   Sentence chunks: 2 chunks
   Paragraph chunks:1 chunks

 Sample: Sushi Rolls
   Fixed chunks:    1 chunks
   Sentence chunks: 2 chunks
   Paragraph chunks:1 chunks

 Sample: Vegetable Pad Thai
   Fixed chunks:    1 chunks
   Sentence chunks: 2 chunks
   Paragraph chunks:1 chunks

 Sample: Shakshuka
   Fixed chunks:    1 chunks
   Sentence chunks: 2 chunks
   Paragraph chunks:1 chunks

 Sample: Mexican Chicken Enchiladas
   Fixed chunks:    1 chunks
   Sentence chunks: 2 chunks
   Paragraph chunks:1 chunks

 Sample: Greek Moussaka
   Fixed chunks:    1 chunks
   Sentence chunks: 2 chunks
   Paragraph chunks:1 chunks

 Sample: Korean Bibimbap
   Fixed chunks:    1 chunks
   Sentence chunks: 

In [5]:
collection_name = "day1_semantic_search"

if client.collection_exists(collection_name=collection_name):
    client.delete_collection(collection_name=collection_name)

# Create a collection with three named vectors
client.create_collection(
    collection_name=collection_name,
    vectors_config={
        "fixed": models.VectorParams(size=384, distance=models.Distance.COSINE),
        "sentence": models.VectorParams(size=384, distance=models.Distance.COSINE),
        "paragraph": models.VectorParams(size=384, distance=models.Distance.COSINE),
    },
)

# Index fields for filtering (more on this on day 2)
client.create_payload_index(collection_name, "chunk_strategy", models.PayloadSchemaType.KEYWORD)
client.create_payload_index(collection_name, "cuisine", models.PayloadSchemaType.KEYWORD)
client.create_payload_index(collection_name, "difficulty", models.PayloadSchemaType.KEYWORD)
# Process and upload data
points = []
point_id = 0

for item in my_dataset:
    description = item["description"]

    # Process with each chunking strategy
    strategies = {
        "fixed": fixed_size_chunks(description),
        "sentence": sentence_chunks(description),
        "paragraph": paragraph_chunks(description),
    }

    for strategy_name, chunks in strategies.items():
        for chunk_idx, chunk in enumerate(chunks):
            # Create vectors for this chunk
            vectors = {strategy_name: encoder.encode(chunk).tolist()}

            points.append(
                models.PointStruct(
                    id=point_id,
                    vector=vectors,
                    payload={
                        **item,  # Include all original metadata
                        "chunk": chunk,
                        "chunk_strategy": strategy_name,
                        "chunk_index": chunk_idx,
                    },
                )
            )
            point_id += 1

client.upload_points(collection_name=collection_name, points=points)
print(f"Uploaded {len(points)} chunks across three strategies")

Uploaded 45 chunks across three strategies


In [6]:
def compare_search(query: str, limit: int = 3):
    """Compare search results for three partitioning strategies using the same query"""
    query_vector = encoder.encode(query).tolist()
 
    print(f"\n{'='*70}")
    print(f"🔍 Query: \"{query}\"")
    print(f"{'='*70}")
 
    for strategy in ["fixed", "sentence", "paragraph"]:
        results = client.query_points(
            collection_name=collection_name,
            query=query_vector,
            using=strategy,
            limit=limit,
        )
        print(f"\n--- {strategy.upper()} CHUNKING ---")
        for i, point in enumerate(results.points, 1):
            title = point.payload["title"]
            score = point.score
            chunk = point.payload["chunk"][:120]
            print(f"  {i}. {title} | Score: {score:.4f}")
            print(f"     Chunk: {chunk}...")
 

In [7]:
compare_search("comfort food for cold winter evenings")
compare_search("spicy sauce with eggs")
compare_search("fresh seafood and clean flavors")
compare_search("slow cooked meat with warm spices")
compare_search("quick and easy weeknight dinner")


🔍 Query: "comfort food for cold winter evenings"

--- FIXED CHUNKING ---
  1. Mexican Chicken Enchiladas | Score: 0.3026
     Chunk: Soft corn tortillas filled with shredded chicken, cheese, and spices, then rolled and baked under a rich chili sauce. Th...
  2. Chicken Tikka Masala | Score: 0.2941
     Chunk: A flavorful dish featuring tender pieces of marinated chicken cooked in a creamy tomato-based curry sauce. The chicken i...
  3. Shakshuka | Score: 0.2620
     Chunk: A vibrant North African and Middle Eastern dish made with eggs gently poached in a spicy tomato and pepper sauce. The sa...

--- SENTENCE CHUNKING ---
  1. Classic Beef Bourguignon | Score: 0.5455
     Chunk: Perfect for cold winter evenings when you want something hearty and 
        comforting. Traditionally served with crust...
  2. Chicken Tikka Masala | Score: 0.4002
     Chunk: Served 
        with steamed basmati rice or warm naan, this dish is comforting, colorful, 
        and perfect for a co...
  3. Sushi

In [8]:
def filtered_search(query: str, cuisine: str = None, difficulty: str = None,
                    strategy: str = "sentence", limit: int = 3):
    """Semantic search with metadata filtering"""
    query_vector = encoder.encode(query).tolist()
 
    conditions = []
    if cuisine:
        conditions.append(FieldCondition(key="cuisine", match=MatchValue(value=cuisine)))
    if difficulty:
        conditions.append(FieldCondition(key="difficulty", match=MatchValue(value=difficulty)))
 
    query_filter = Filter(must=conditions) if conditions else None
 
    print(f"\n{'='*70}")
    print(f"🔍 Query: \"{query}\" | Filter: cuisine={cuisine}, difficulty={difficulty}")
    print(f"{'='*70}")
 
    results = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        using=strategy,
        query_filter=query_filter,
        limit=limit,
    )
    print(f"\n--- {strategy.upper()} + FILTER ---")
    for i, point in enumerate(results.points, 1):
        p = point.payload
        print(f"  {i}. {p['title']} ({p['cuisine']}, {p['difficulty']}) | Score: {point.score:.4f}")
        print(f"     Chunk: {p['chunk'][:120]}...")

In [9]:
filtered_search("creamy rich sauce", cuisine="Italian")
filtered_search("warm spicy dish", difficulty="Easy")


🔍 Query: "creamy rich sauce" | Filter: cuisine=Italian, difficulty=None

--- SENTENCE + FILTER ---
  1. Spaghetti Carbonara (Italian, Intermediate) | Score: 0.5566
     Chunk: Best served 
        immediately while the sauce is glossy and the pasta is perfectly al dente....
  2. Spaghetti Carbonara (Italian, Intermediate) | Score: 0.4292
     Chunk: A classic Roman pasta dish made with silky eggs, 
        sharp Pecorino Romano cheese, crispy guanciale, and freshly cr...

🔍 Query: "warm spicy dish" | Filter: cuisine=None, difficulty=Easy

--- SENTENCE + FILTER ---
  1. Shakshuka (Middle Eastern, Easy) | Score: 0.4778
     Chunk: A vibrant North African and Middle Eastern dish made 
        with eggs gently poached in a spicy tomato and pepper sauc...
  2. Shakshuka (Middle Eastern, Easy) | Score: 0.3149
     Chunk: Shakshuka is often served 
        straight from the pan with crusty bread for dipping. It works wonderfully 
        as...


In [10]:
def analyze_chunking():
    """Calculate the chunk count and length distribution for each strategy."""
    print(f"\n{'='*70}")
    print(f"📊 Chunking Analysis")
    print(f"{'='*70}")
 
    for strategy in ["fixed", "sentence", "paragraph"]:
        results = client.scroll(
            collection_name=collection_name,
            scroll_filter=Filter(
                must=[FieldCondition(key="chunk_strategy", match=MatchValue(value=strategy))]
            ),
            limit=500,
        )[0]
 
        lengths = [len(p.payload["chunk"].split()) for p in results]
        total = len(lengths)
        avg_len = sum(lengths) / total if total else 0
        min_len = min(lengths) if lengths else 0
        max_len = max(lengths) if lengths else 0
 
        print(f"\n  {strategy.upper()}:")
        print(f"    Total chunks: {total}")
        print(f"    Avg length:   {avg_len:.1f} words")
        print(f"    Range:        {min_len} - {max_len} words")

In [11]:
analyze_chunking()


📊 Chunking Analysis

  FIXED:
    Total chunks: 12
    Avg length:   69.2 words
    Range:        1 - 81 words

  SENTENCE:
    Total chunks: 22
    Avg length:   37.7 words
    Range:        14 - 66 words

  PARAGRAPH:
    Total chunks: 11
    Avg length:   75.5 words
    Range:        68 - 81 words


In [12]:
def analyze_chunking_effectiveness():
    """Analyze which chunking strategy works best for your domain"""

    print("CHUNKING STRATEGY ANALYSIS")
    print("=" * 40)

    # Get chunk statistics for each strategy
    for strategy in ["fixed", "sentence", "paragraph"]:
        # Count chunks per strategy
        results = client.scroll(
            collection_name=collection_name,
            scroll_filter=models.Filter(
                must=[
                    models.FieldCondition(
                        key="chunk_strategy", match=models.MatchValue(value=strategy)
                    )
                ]
            ),
            limit=100,
        )

        chunks = results[0]
        chunk_sizes = [len(chunk.payload["chunk"]) for chunk in chunks]

        print(f"\n{strategy.upper()} STRATEGY:")
        print(f"  Total chunks: {len(chunks)}")
        print(f"  Avg chunk size: {sum(chunk_sizes)/len(chunk_sizes):.0f} chars")
        print(f"  Size range: {min(chunk_sizes)}-{max(chunk_sizes)} chars")


analyze_chunking_effectiveness()

CHUNKING STRATEGY ANALYSIS

FIXED STRATEGY:
  Total chunks: 12
  Avg chunk size: 435 chars
  Size range: 6-505 chars

SENTENCE STRATEGY:
  Total chunks: 22
  Avg chunk size: 263 chars
  Size range: 94-468 chars

PARAGRAPH STRATEGY:
  Total chunks: 11
  Avg chunk size: 530 chars
  Size range: 491-568 chars
